# 调节效应汇总图

精选显著交互作用的 Johnson-Neyman 网格图和 cluster 调节效应汇总图。
用于论文发表。Helper 函数与 03c_moderation.ipynb 完全一致。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

corr_labels_cn = {
    'trial_calls': 'AI调用次数',
    'first_ai_time': '首次调用时间',
    'pre_first_call_ideas': '首次调用前想法数',
    'pre_think_time': '平均调用前思考时间',
    'perspective_taking': '观点采纳率',
    'affected_by_ai': '受AI影响率',
    'bfi_extroversion': '外向性',
    'bfi_agreeableness': '宜人性',
    'bfi_conscientiousness': '尽责性',
    'bfi_neuroticism': '神经质',
    'bfi_openness': '开放性',
    'cse_total': '创造性自我效能',
    'ribs_total': 'RIBS总分',
    'ai_attitude': 'AI态度',
    'dat_score': 'DAT分数',
    'originality': '原创性',
    'fluency': '流畅性',
    'above_median': '高质量回答数',
    'above_median_ratio': '高质量回答率',
    'cluster': '簇别',
    'age': '年龄',
    'gender': '性别',
}

# 输入文件路径
output_dir = Path('output')
trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'
# 加载数据
df_cluster = pd.read_csv(trial_with_cluster_file)
df_perf = pd.read_csv(performance_file)

# 以试次为单位合并聚类标签（保持原始 left join）
df_merged = pd.merge(
    df_perf,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
)

print(df_merged['cluster'].value_counts(dropna=False).sort_index())

# 准备调节分析数据
mod_df = df_merged[df_merged["cluster"] >= 0].copy()

def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

# 对所有参与分析的连续变量做 Z 标准化
for col in ["trial_calls", "first_ai_time", "pre_first_call_ideas", "pre_think_time",
            "cse_total", "ribs_total", "dat_score", "bfi_openness", "ai_attitude",
            "perspective_taking", "affected_by_ai", "age",
            "originality", "fluency"]:
    if col in mod_df.columns:
        mod_df[f"{col}_z"] = zscore(mod_df[col])

print(f"mod_df 样本量: {len(mod_df)}")

In [ ]:
# ---- 核心拟合函数 ----
# 与 03c_moderation_clean.ipynb 保持一致

def _fit_mixed_or_ols(formula, data, has_groups):
    """尝试 LMM，失败则回退至 OLS。"""
    if has_groups:
        try:
            model = sm.MixedLM.from_formula(
                formula,
                groups="participant_id",
                vc_formula={"item_name": "0 + C(item_name)"},
                data=data
            )
            result = model.fit(reml=False, method="lbfgs", maxiter=2000)
            return result, "LMM"
        except Exception:
            pass
    model = ols(formula, data=data).fit()
    return model, "OLS"


def _fit_and_jn(predictor, moderator, outcome, data, controls=None, alpha=0.05, verbose=False):
    """拟合交互模型，返回 Johnson-Neyman 简单斜率数据。"""
    if controls is None:
        controls = []

    p_z = f"{predictor}_z" if f"{predictor}_z" in data.columns else predictor
    m_z = f"{moderator}_z" if f"{moderator}_z" in data.columns else moderator

    subset = data.dropna(subset=[p_z, m_z, outcome] + controls).copy()
    if len(subset) < 30:
        return None

    ctrl_str = " + ".join(controls) if controls else ""
    rhs = f"{p_z} * {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
    formula = f"{outcome} ~ {rhs}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    model, mtype = _fit_mixed_or_ols(formula, subset, has_groups)

    # 提取系数与协方差
    if mtype == "LMM":
        params = model.fe_params
        cov_fe = model.cov_params().loc[params.index, params.index]
        main_formula = f"{outcome} ~ {p_z} + {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
        try:
            model_main, _ = _fit_mixed_or_ols(main_formula, subset, has_groups)
            lr_stat = 2 * (model.llf - model_main.llf)
            p_inter = stats.chi2.sf(lr_stat, 1)
        except Exception:
            p_inter = np.nan
    else:
        params = model.params
        cov_fe = model.cov_params()
        if hasattr(cov_fe, "loc"):
            cov_fe = cov_fe.loc[params.index, params.index]
        main_formula = f"{outcome} ~ {p_z} + {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
        model_main = ols(main_formula, data=subset).fit()
        anova_res = sm.stats.anova_lm(model_main, model)
        p_inter = anova_res["Pr(>F)"].iloc[1] if "Pr(>F)" in anova_res.columns else np.nan

    # 定位交互项
    inter_terms = [k for k in params.index if ":" in k and p_z in k]
    if not inter_terms:
        inter_terms = [k for k in params.index if ":" in k]
    if not inter_terms:
        return None
    inter_key = inter_terms[0]

    b_pred = params.get(p_z)
    b_inter = params.get(inter_key)
    if b_pred is None or b_inter is None:
        return None

    try:
        v_pred = cov_fe.at[p_z, p_z]
        v_inter = cov_fe.at[inter_key, inter_key]
        cov_pi = cov_fe.at[p_z, inter_key]
    except KeyError:
        return None

    # ---- JN 计算 ----
    z_crit = stats.norm.ppf(1 - alpha / 2)

    m_raw = subset[moderator]
    m_model = subset[m_z]
    m_raw_min, m_raw_max = m_raw.min(), m_raw.max()
    m_model_min, m_model_max = m_model.min(), m_model.max()

    m_range = np.linspace(m_raw_min, m_raw_max, 300)
    m_model_range = np.linspace(m_model_min, m_model_max, 300)

    simple_slope = b_pred + b_inter * m_model_range
    var_slope = v_pred + m_model_range**2 * v_inter + 2 * m_model_range * cov_pi
    se_slope = np.sqrt(np.maximum(var_slope, 0))
    ci_lo = simple_slope - z_crit * se_slope
    ci_hi = simple_slope + z_crit * se_slope
    sig_mask = (ci_lo > 0) | (ci_hi < 0)

    # 解 JN 边界点
    A = b_inter**2 - z_crit**2 * v_inter
    B = 2 * (b_pred * b_inter - z_crit**2 * cov_pi)
    C = b_pred**2 - z_crit**2 * v_pred

    jn_points_raw = []
    if abs(A) > 1e-12:
        disc = B**2 - 4 * A * C
        if disc >= 0:
            sqrt_disc = np.sqrt(disc)
            for root_model in [(-B + sqrt_disc) / (2 * A), (-B - sqrt_disc) / (2 * A)]:
                if m_model_min <= root_model <= m_model_max:
                    root_raw = np.interp(root_model, [m_model_min, m_model_max], [m_raw_min, m_raw_max])
                    jn_points_raw.append(root_raw)

    if verbose:
        print(f"  [{predictor} x {moderator} -> {outcome}] {mtype}, p_inter={p_inter:.4f}")
        print(f"    斜率范围: [{simple_slope.min():.3f}, {simple_slope.max():.3f}]")
        print(f"    显著比例: {sig_mask.mean():.1%}")
        if jn_points_raw:
            print(f"    JN 边界: {jn_points_raw}")

    return {
        "m_range": m_range, "simple_slope": simple_slope,
        "ci_lo": ci_lo, "ci_hi": ci_hi, "sig_mask": sig_mask,
        "jn_points": sorted(jn_points_raw), "p_inter": p_inter,
        "p_cn": corr_labels_cn.get(predictor, predictor),
        "m_cn": corr_labels_cn.get(moderator, moderator),
        "y_cn": corr_labels_cn.get(outcome, outcome),
        "mtype": mtype, "m_raw": subset[moderator],
        "p_raw": subset[p_z], "y_raw": subset[outcome],
        "m_mean": m_raw.mean(), "m_sd": m_raw.std(ddof=0)
    }

In [ ]:
# ---- 汇总图函数 ----

def plot_moderation_grid_jn(interactions, data, n_cols=3, controls=None, figsize_per_cell=(4.5, 3.5)):
    """连续调节变量 JN 网格汇总图。"""
    if controls is None:
        controls = ["age", "gender"]

    valid = []
    for pred, mod, out in interactions:
        res = _fit_and_jn(pred, mod, out, data, controls)
        if res is not None:
            valid.append(res)

    n_rows = (len(valid) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_cell[0] * n_cols, figsize_per_cell[1] * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, res in enumerate(valid):
        ax = axes[i]

        if res["sig_mask"].any():
            in_sig = False; start = None
            for j, sig in enumerate(res["sig_mask"]):
                if sig and not in_sig:
                    start = res["m_range"][j]
                    in_sig = True
                elif not sig and in_sig:
                    ax.axvspan(start, res["m_range"][j], color="#FFF3B0", alpha=0.6, lw=0)
                    in_sig = False
            if in_sig:
                ax.axvspan(start, res["m_range"][-1], color="#FFF3B0", alpha=0.6, lw=0)

        ax.fill_between(res["m_range"], res["ci_lo"], res["ci_hi"], color="#2c7bb6", alpha=0.10, lw=0)
        ax.plot(res["m_range"], res["simple_slope"], color="#2c7bb6", linewidth=2)
        ax.axhline(y=0, color="gray", linestyle=":", linewidth=0.8)

        for jp in res["jn_points"]:
            ax.axvline(x=jp, color="#D4A017", linestyle="--", linewidth=1, alpha=0.8)

        ax.set_xlabel(res["m_cn"], fontsize=9)
        ax.set_ylabel(f'{res["p_cn"]} 的简单斜率', fontsize=9)
        ax.set_title(f'{res["p_cn"]} x {res["m_cn"]} → {res["y_cn"]}', fontsize=10, fontweight="bold")

    for j in range(len(valid), len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()
    return fig


def _fit_cat_interaction(predictor, outcome, moderator="cluster", data=None,
                         controls=None, alpha=0.05):
    """拟合分类调节模型，返回每个 cluster 水平的预测线及斜率信息。"""
    import patsy as _patsy

    if data is None:
        data = mod_df
    if controls is None:
        controls = ["age", "gender"]

    p_z = f"{predictor}_z" if f"{predictor}_z" in data.columns else predictor

    subset = data.dropna(subset=[p_z, moderator, outcome] + controls).copy()
    subset[moderator] = subset[moderator].astype("category")
    levels = list(subset[moderator].cat.categories)

    ctrl_terms = []
    for c in controls:
        if c not in subset.columns:
            continue
        if pd.api.types.is_numeric_dtype(subset[c]):
            ctrl_terms.append(c)
        else:
            ctrl_terms.append(f"C({c})")
    ctrl_str = " + ".join(ctrl_terms) if ctrl_terms else ""

    rhs = f"{p_z} * C({moderator})"
    if ctrl_str:
        rhs += " + " + ctrl_str
    formula = f"{outcome} ~ {rhs}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    model, mtype = _fit_mixed_or_ols(formula, subset, has_groups)

    # 提取参数与协方差
    if mtype == "LMM":
        params = model.fe_params
        cov_raw = model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov_fe = cov_raw.loc[params.index, params.index]
        else:
            tmp = pd.DataFrame(cov_raw, index=model.model.exog_names,
                               columns=model.model.exog_names)
            cov_fe = tmp.loc[params.index, params.index]

        main_rhs = f"{p_z} + C({moderator})"
        if ctrl_str:
            main_rhs += " + " + ctrl_str
        main_formula = f"{outcome} ~ {main_rhs}"
        try:
            model_main, _ = _fit_mixed_or_ols(main_formula, subset, has_groups)
            lr_stat = 2 * (model.llf - model_main.llf)
            df_diff = len(params) - len(model_main.fe_params)
            p_inter = stats.chi2.sf(lr_stat, df_diff)
        except Exception:
            p_inter = np.nan
    else:
        params = model.params
        cov_raw = model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov_fe = cov_raw.loc[params.index, params.index]
        else:
            cov_fe = pd.DataFrame(cov_raw, index=params.index, columns=params.index)

        main_rhs = f"{p_z} + C({moderator})"
        if ctrl_str:
            main_rhs += " + " + ctrl_str
        main_model = ols(f"{outcome} ~ {main_rhs}", data=subset).fit()
        anova_res = sm.stats.anova_lm(main_model, model)
        p_inter = anova_res["Pr(>F)"].iloc[1]

    # 预测
    x_vals = subset[p_z]
    x_range = np.linspace(x_vals.min(), x_vals.max(), 100)

    ctrl_baseline = {}
    for c in controls:
        if c not in subset.columns:
            continue
        if pd.api.types.is_numeric_dtype(subset[c]):
            ctrl_baseline[c] = subset[c].mean()
        else:
            ctrl_baseline[c] = subset[c].mode().iloc[0]

    colors = plt.cm.Set2(np.linspace(0, 1, len(levels)))
    z_crit = stats.norm.ppf(1 - alpha / 2)
    ref_level = levels[0]

    level_lines = []
    slope_info = []

    for i_lvl, level in enumerate(levels):
        pred_df = pd.DataFrame({
            p_z: x_range,
            moderator: pd.Categorical([level] * len(x_range), categories=levels),
        })
        for c, v in ctrl_baseline.items():
            pred_df[c] = v

        design = _patsy.dmatrix(rhs, pred_df, return_type="dataframe")
        for cn in params.index:
            if cn not in design.columns:
                design[cn] = 0.0
        design = design[params.index]

        cov_v = cov_fe.values
        y_pred = design.values @ params.values
        var_pred = np.sum((design.values @ cov_v) * design.values, axis=1)
        se_pred = np.sqrt(np.maximum(var_pred, 0))
        ci_lo = y_pred - z_crit * se_pred
        ci_hi = y_pred + z_crit * se_pred

        label = cluster_name_map.get(int(level), f"簇 {int(level)}")
        level_lines.append({
            "level": level, "label": label, "color": colors[i_lvl],
            "y_pred": y_pred, "ci_lo": ci_lo, "ci_hi": ci_hi,
        })

        slope_val = params.get(p_z, 0)
        inter_key = f"{p_z}:C({moderator})[T.{level}]"
        if level != ref_level:
            slope_val += params.get(inter_key, 0)

        contrast = pd.Series(0.0, index=params.index)
        contrast[p_z] = 1.0
        if level != ref_level and inter_key in contrast.index:
            contrast[inter_key] = 1.0

        var_slope = float(contrast.values @ cov_fe.values @ contrast.values)
        se_slope = np.sqrt(max(var_slope, 0))
        slope_info.append({
            "slope": slope_val, "se": se_slope,
            "ci_lo": slope_val - z_crit * se_slope,
            "ci_hi": slope_val + z_crit * se_slope,
        })

    return {
        "x_range": x_range, "x_raw": subset[p_z], "y_raw": subset[outcome],
        "level_lines": level_lines, "slope_info": slope_info,
        "p_inter": p_inter, "mtype": mtype,
        "p_cn": corr_labels_cn.get(predictor, predictor),
        "y_cn": corr_labels_cn.get(outcome, outcome),
    }


def plot_cluster_moderation_grid(interactions, data=None, n_cols=4,
                                  controls=None, figsize_per_cell=(2.8, 2.8)):
    """分类调节变量 (cluster) 交互效应网格图。"""
    if data is None:
        data = mod_df
    if controls is None:
        controls = ["age", "gender"]

    valid = []
    for pred, out in interactions:
        res = _fit_cat_interaction(pred, out, data=data, controls=controls)
        if res is not None:
            valid.append(res)
            print(f"  {pred} -> {out}: p_inter = {res['p_inter']:.4f}")
        else:
            print(f"  !! 跳过 {pred} -> {out}")

    n = len(valid)
    if n == 0:
        print("无有效结果")
        return None

    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_cell[0] * n_cols,
                                      figsize_per_cell[1] * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, res in enumerate(valid):
        ax = axes[i]

        ax.scatter(res["x_raw"], res["y_raw"], color=".85", s=3, alpha=0.35,
                   edgecolors="none")

        for ll in res["level_lines"]:
            ax.plot(res["x_range"], ll["y_pred"], color=ll["color"], linewidth=2,
                    label=ll["label"])
            ax.fill_between(res["x_range"], ll["ci_lo"], ll["ci_hi"],
                            color=ll["color"], alpha=0.08)

        ax.set_xlabel(res["p_cn"], fontsize=9)
        ax.set_ylabel(res["y_cn"], fontsize=9)
        ax.legend(fontsize=7, loc="upper left", framealpha=0.8)
        ax.set_title(res["p_cn"] + " x 簇别 → " + res["y_cn"],
                     fontsize=9, fontweight="bold")

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()
    return fig

In [ ]:
# ===== 人格特质 JN 汇总图 =====
sig_trait_interactions = [
    ("first_ai_time",      "cse_total",    "fluency"),
    ("pre_think_time",     "ribs_total",   "fluency"),
    ("pre_think_time",     "ribs_total",   "above_median"),
    ("trial_calls",        "ribs_total",   "originality"),
    ("pre_think_time",     "dat_score",    "originality"),
    ("pre_think_time",     "dat_score",    "above_median_ratio"),
    ("perspective_taking", "dat_score",    "originality"),
    ("perspective_taking", "dat_score",    "above_median_ratio"),
]

print("开始生成汇总 JN 调节效应图...")
fig = plot_moderation_grid_jn(sig_trait_interactions, mod_df, n_cols=3)
fig.savefig("output/moderation_personality_jn_clean.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ===== Cluster 调节效应汇总图 =====
cluster_interactions = [
    # 流畅性
    ("trial_calls",          "fluency"),
    ("first_ai_time",        "fluency"),
    ("pre_first_call_ideas", "fluency"),
    ("pre_think_time",       "fluency"),
    ("perspective_taking",   "fluency"),
    ("affected_by_ai",       "fluency"),
    # 高质量回答数
    ("trial_calls",          "above_median"),
    ("first_ai_time",        "above_median"),
    ("pre_first_call_ideas", "above_median"),
    ("pre_think_time",       "above_median"),
    ("perspective_taking",   "above_median"),
    ("affected_by_ai",       "above_median"),
    # 原创性
    ("first_ai_time",        "originality"),
]

print("开始生成 Cluster 调节效应汇总图...")
fig_cluster = plot_cluster_moderation_grid(cluster_interactions, mod_df, n_cols=4)
fig_cluster.savefig("output/moderation_cluster_summary.png", dpi=200, bbox_inches="tight")
plt.show()